# Protocolo Titán — Notebook replanteado

Análisis reproducible de los escenarios A y B.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from protocolo_titan.config import GSMConfig, ConvoyScenario, CampScenario, AnalyzerConfig
from protocolo_titan.radio_access import gsm_access_summary
from protocolo_titan.scenario_a import analyze_convoy_mobility, analyze_convoy_fading
from protocolo_titan.scenario_b import analyze_camp_base


## 1. Capa física: FDMA y TDMA

In [ ]:
gsm = GSMConfig()
gsm_access_summary(gsm)


## 2. Escenario A: convoy de alta velocidad

In [ ]:
convoy = ConvoyScenario()
mobility = analyze_convoy_mobility(convoy, gsm)
mobility


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))
plt.bar(mobility["speed_kmh"].astype(str), mobility["coherence_time_ms"])
plt.axhline(mobility["gsm_timeslot_ms"].iloc[0], linestyle="--", label="Timeslot GSM")
plt.xlabel("Velocidad (km/h)")
plt.ylabel("Tiempo (ms)")
plt.title("Tiempo de coherencia vs timeslot GSM")
plt.grid(True, axis="y")
plt.legend()
plt.show()


## 3. Rayleigh y Rician durante el timeslot

In [ ]:
fading_summary, traces = analyze_convoy_fading(convoy, gsm)
fading_summary


In [ ]:
trace = traces["rayleigh_250_kmh"]

plt.figure(figsize=(8,4))
plt.plot(trace["time_us"], trace["envelope_normalized"])
plt.xlabel("Tiempo dentro del timeslot (µs)")
plt.ylabel("Envolvente normalizada")
plt.title("Fading Rayleigh a 250 km/h durante 577 µs")
plt.grid(True)
plt.show()


## 4. Escenario B: campamento base

In [ ]:
camp = CampScenario()
analyzer = AnalyzerConfig()
results_b = analyze_camp_base(camp, gsm, analyzer)
results_b["frequency_planning"]


In [ ]:
results_b["logical_channels"].head(12)


## 5. Certificación e instrumentación: RBW

In [ ]:
results_b["rbw_noise"]


In [ ]:
noise = results_b["rbw_noise"]

plt.figure(figsize=(7,4))
plt.plot(noise["rbw_khz"], noise["noise_floor_dbm"], marker="o")
plt.xscale("log")
plt.xlabel("RBW (kHz)")
plt.ylabel("Suelo de ruido (dBm)")
plt.title("Ruido integrado del analizador frente a RBW")
plt.grid(True)
plt.show()


## 6. Checklist RED

In [ ]:
results_b["red_checklist"]
